# LLM population experiment
Default: SmolLM2 actor with private LoRA, separate critic, recurrent PPO. Training remains off until you set `RUN = True`. See `docs/next-experiment.md` for the hypotheses, limits, and measured predator difficulty axis. The CPU sweep may need several overnight invocations.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / 'src/millstlabs').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src/millstlabs').exists(), 'Open this notebook from the millstlabs checkout'
os.chdir(ROOT)
os.environ.setdefault('HF_HOME', str(ROOT / '.cache/huggingface'))
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
PHASE = 'core'  # core, communication, muted, shared; numeric is explicitly non-LLM
HOURS = 10
RUN = False
from millstlabs.config import load_config
from millstlabs.batch import plan
cfg = load_config(f'configs/next-{PHASE}.yaml')
print(json.dumps(plan(cfg), indent=2))

In [ ]:
if RUN:
    subprocess.run([sys.executable, '-m', 'millstlabs.next_experiment', '--phase', PHASE,
                    '--mode', 'run', '--hours', str(HOURS)], check=True)
else:
    print('Training is disabled. Set RUN = True to start/resume this phase.')

In [ ]:
from millstlabs.batch import report
root = ROOT / 'runs' / ('next-numeric' if PHASE == 'numeric' else 'next-llm-' + PHASE)
if root.exists():
    status = report(cfg, root)
    print(f"Completed {status['complete']}/{status['total']}")
    for r in status['runs']:
        print(r['profile'], r['method'], r['status'], r['decisions'], r.get('final_lifetime'))
else:
    print('No runs in this phase yet.')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for directory in sorted(root.glob('*-s11')):
    p = directory / 'evaluation.jsonl'
    if p.exists():
        rows = list({r['checkpoint']: r for r in map(json.loads, p.read_text().splitlines())}.values())
        axes[0].plot([r['checkpoint'] for r in rows], [r['restricted_mean_lifetime'] for r in rows], label=directory.name)
    p = directory / 'predator_evaluation.jsonl'
    if p.exists():
        rows = list({r['predator_temperature']: r for r in map(json.loads, p.read_text().splitlines())}.values())
        rows.sort(key=lambda r: r['predator_temperature'])
        axes[1].plot([r['predator_temperature'] for r in rows], [r['restricted_mean_lifetime'] for r in rows], marker='o', label=directory.name)
axes[0].set(xlabel='Training decisions', ylabel='Restricted mean lifetime', title='Fixed task: LLM learning')
axes[1].set(xlabel='Predator temperature', ylabel='Restricted mean lifetime', title='Frozen policy: uncertainty curve')
for ax in axes:
    if ax.lines: ax.legend(fontsize=6)
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()